# Progetto Smart Vehicular Systems - Multi-modal black box recorder
  - Gabriele Arcese
  - Diego Colì

## Setup

In [1]:
%pip install paho-mqtt keyboard

     ---------------------------------------- 58.1/58.1 kB 3.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


### Restart kernel!

In [1]:
import carla
import os
import shutil
import time
import random
import socket
import threading
import queue
import numpy as np
import json
import paho.mqtt.client as mqtt
from dataclasses import dataclass, field, asdict
from typing import Optional
from collections import deque


try:
    import pygame
except ImportError:
    pygame = None
    
try:
    import cv2
    import numpy as np
except ImportError:
    cv2 = None
    np = None

pygame 2.6.1 (SDL 2.28.4, Python 3.7.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


## Connessione al client

In [2]:
client = carla.Client("localhost", 2000)
client.set_timeout(15.0)
world = client.get_world()
spectator = world.get_spectator()

print(f"Client version: {client.get_client_version()}")
print(f"Server version: {client.get_server_version()}")

Client version: 0.9.15
Server version: 0.9.15


## MQTT client

In [3]:
# queue thread-safe per accumulare messaggi ricevuti
mqtt_msgs_q = queue.Queue()
ring_buffer = deque(maxlen=600)  

def _on_connect(client, userdata, flags, rc):
    if rc == 0:
        client.connected_flag = True
        # sottoscrivi i topic desiderati
        client.subscribe([("recorder/control", 0), ("vehicle/+/status", 0)])
    else:
        client.connected_flag = False

def _on_message(client, userdata, msg):
    ts = time.time()
    payload = msg.payload
    # prova a decodificare json, fallback a stringa/bytes
    parsed = None
    try:
        parsed = json.loads(payload.decode("utf-8"))
    except Exception:
        try:
            parsed = payload.decode("utf-8")
        except Exception:
            parsed = payload  # raw bytes
    record = {"topic": msg.topic, "payload": parsed, "qos": msg.qos, "ts": ts}
    try:
        mqtt_msgs_q.put_nowait(record)
    except queue.Full:
        pass

def start_mqtt(broker="localhost", port=1883, keepalive=60):
    client = mqtt.Client()
    client.on_connect = _on_connect
    client.on_message = _on_message
    client.connected_flag = False
    client.connect(broker, port, keepalive=keepalive)
    client.loop_start()   # avvia loop in background thread
    return client

def stop_mqtt(client):
    if client is None: return
    try:
        client.loop_stop()
        client.disconnect()
    except Exception:
        pass

# Helper: svuota la queue e ritorna lista (da chiamare nel tick)
def drain_mqtt_queue():
    items = []
    while True:
        try:
            items.append(mqtt_msgs_q.get_nowait())
        except queue.Empty:
            break
    return items

## Dataclass EventRecord
Prodotto ad ogni tick della simulazione: è l'unità atomica del ring buffer.

| Campo | Sorgente CARLA | Tipo |
|---|---|---|
| `timestamp_sim` | `world.get_snapshot().timestamp.elapsed_seconds` | `float` |
| `timestamp_wall` | `time.time()` | `float` |
| `frame_id` | contatore incrementale | `int` |
| `camera_frames` | callback `sensor.camera.rgb` (una per camera) | `dict[str, np.ndarray]` |
| `location` | `vehicle.get_location()` | `carla.Location` |
| `velocity` | `vehicle.get_velocity()` | `carla.Vector3D` |
| `acceleration` | `vehicle.get_acceleration()` | `carla.Vector3D` |
| `control` | `vehicle.get_control()` | `carla.VehicleControl` |
| `warnings` | logica interna (LiDAR, collisione) | `list[str]` |
| `mqtt_messages` | thread MQTT | `list[str]` |
| `reasoning_text` | explanation agent | `str \| None` |
| `reasoning_ts` | wall-clock emissione agent | `float \| None` |

In [4]:
@dataclass
class VehicleState:
    """Stato cinematico e di controllo del veicolo in un istante."""
    x: float
    y: float
    z: float
    vx: float
    vy: float
    vz: float
    ax: float
    ay: float
    az: float
    throttle: float
    brake: float
    steer: float
    hand_brake: bool
    reverse: bool

    @property
    def speed_ms(self) -> float:
        """Velocità scalare in m/s."""
        return float(np.sqrt(self.vx**2 + self.vy**2 + self.vz**2))

    @property
    def speed_kmh(self) -> float:
        return self.speed_ms * 3.6

@dataclass
class EventRecord:
    """Unità atomica del ring buffer — un record per tick di simulazione."""
    # --- Temporali ---
    frame_id: int            # contatore tick, usato come chiave nel viewer
    timestamp_sim: float     # secondi simulazione (univoco per tick in sync mode)
    timestamp_wall: float    # wall-clock al momento della costruzione

    # --- Percezione ---
    vehicle_state: VehicleState

    # --- Camera (dict role → frame BGR; vuoto se nessun frame è ancora arrivato) ---
    # Esempio: {"front": np.ndarray, "rear": np.ndarray}
    camera_frames: dict = field(default_factory=dict, repr=False)

    # --- Logica e messaggistica ---
    warnings: list = field(default_factory=list)       # es. ["COLLISION_IMMINENT"]
    mqtt_messages: list = field(default_factory=list)  # payload raw ricevuti nel tick

    # --- Explanation agent ---
    reasoning_text: Optional[str] = None
    reasoning_ts: Optional[float] = None  # wall-clock di emissione dell'agente

    def to_dict(self) -> dict:
        """Serializza tutto tranne camera_frames (salvati separatamente come JPEG)."""
        d = asdict(self)
        d.pop("camera_frames")  # i frame vanno in frames/<frame_id:06d>_<role>.jpg
        return d

    def has_camera(self, role: str = None) -> bool:
        """True se almeno una camera (o la camera `role`) ha un frame nel tick."""
        if role:
            return role in self.camera_frames
        return len(self.camera_frames) > 0

    def camera_roles(self) -> list:
        return list(self.camera_frames.keys())

    def has_reasoning(self) -> bool:
        return self.reasoning_text is not None

Il **timestamp** di simulazione è la chiave di allineamento: tutto viene sincronizzato su di esso perché in modalità sincrona.

## VehicleState

In [5]:
# VehicleState partendo da un attore
def build_vehicle_state(vehicle) -> VehicleState:
    loc = vehicle.get_location()
    vel = vehicle.get_velocity()
    acc = vehicle.get_acceleration()
    ctrl = vehicle.get_control()
    return VehicleState(
    x=loc.x, y=loc.y, z=loc.z,
    vx=vel.x, vy=vel.y, vz=vel.z,
    ax=acc.x, ay=acc.y, az=acc.z,
    throttle=ctrl.throttle,
    brake=ctrl.brake,
    steer=ctrl.steer,
    hand_brake=ctrl.hand_brake,
    reverse=ctrl.reverse,
)

## Ciclo di simulazione e connessione MQTT

In [6]:
def tcp_check(host, port, timeout=2.0):
    s = socket.socket()
    s.settimeout(timeout)
    try:
        s.connect((host, port))
        s.close()
        return True, None
    except Exception as e:
        return False, str(e)

targets = [("localhost", 2001), ("test.mosquitto.org", 1883)]
for host, port in targets:
    ok, err = tcp_check(host, port)
    print(f"{host}:{port} -> {'OK' if ok else 'FAIL'}{'' if ok else ': '+err}")

# Try connecting with paho to localhost (shows MQTT error if refused)
client = mqtt.Client()
try:
    client.connect("localhost", 2001, keepalive=5)
    client.loop_start()
    time.sleep(0.3)
    client.loop_stop()
    print("paho-mqtt: connection to localhost:2001 succeeded")
except Exception as e:
    print("paho-mqtt: connection to localhost:2001 failed:", e)
    print("\nHints:")
    print("- If you want a local broker, start Mosquitto (outside notebook) or run Docker:")
    print("  docker run -d --name mosquitto -p 2001:2001 -p 9001:9001 eclipse-mosquitto")
    print("- Or switch to the public test broker: test.mosquitto.org:1883")

localhost:2001 -> OK
test.mosquitto.org:1883 -> FAIL: timed out


c:\anaconda3\envs\carla-env\lib\site-packages\ipykernel_launcher.py:17: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  app.launch_new_instance()


paho-mqtt: connection to localhost:2001 succeeded


## Attivazione modalità sync
In modalità **sincrona**, il server avanza solo quando il client chiama `world.tick()`. Ciò è utile per esperimenti controllati e risultati riproducibili.


In [29]:
settings = world.get_settings()
settings.synchronous_mode = True
world.apply_settings(settings)
print("Modalità sincrona attivata")

Modalità sincrona attivata


## Preparazione veicolo e target

In [30]:
def move_spectator_to(transform, spectator, distance=7.0, x=0, y=0, z=3.0, yaw=0, pitch=-15, roll=0):
    back_location = transform.location - transform.get_forward_vector() * distance
    
    back_location.x += x
    back_location.y += y
    back_location.z += z
    transform.rotation.yaw += yaw
    transform.rotation.pitch = pitch
    transform.rotation.roll = roll
    
    spectator_transform = carla.Transform(back_location, transform.rotation)
    
    spectator.set_transform(spectator_transform)

def spawn_vehicle(world, vehicle_index=20, spawn_index=10):
    blueprint_library = world.get_blueprint_library()
    vehicle_bp = blueprint_library.filter('vehicle.*')[vehicle_index]
    spawn_point = world.get_map().get_spawn_points()[spawn_index]
    vehicle = world.spawn_actor(vehicle_bp, spawn_point)
    return vehicle

def draw_on_screen(world, transform, content="O", color=carla.Color(0, 255, 0), life_time=20):
    world.debug.draw_string(transform.location, content, color=color, life_time=life_time)

def spawn_vehicle_ahead(world, ref_vehicle, distances=(20.0, 28.0, 36.0),
                        same_lane_first=True, retries=6):
    """Spawna un veicolo davanti al veicolo di riferimento."""
    ego_tf  = ref_vehicle.get_transform()
    ego_loc = ego_tf.location
    fwd     = ego_tf.get_forward_vector()
    bps     = world.get_blueprint_library().filter("vehicle.*")

    # Phase 1: offset diretto
    direct = sorted(set(float(d) for d in distances))
    direct += [d + 4.0 for d in direct]
    for d in direct:
        loc = ego_loc + fwd * d
        loc.z += 0.30
        actor = world.try_spawn_actor(random.choice(bps),
                                      carla.Transform(loc, ego_tf.rotation))
        if actor is not None:
            actor.set_autopilot(False)
            return actor

    # Phase 2: waypoint davanti
    road_map = world.get_map()
    ego_wp   = road_map.get_waypoint(ego_loc, project_to_road=True,
                                     lane_type=carla.LaneType.Driving)
    wp_candidates = []
    d_pool = sorted(set(float(d) for d in distances) |
                    set(float(d) + 8.0 for d in distances))
    for d in d_pool:
        try:
            cands = ego_wp.next(float(d))
        except RuntimeError:
            cands = []
        if same_lane_first:
            cands = sorted(cands, key=lambda w: (
                w.road_id != ego_wp.road_id,
                w.lane_id != ego_wp.lane_id,
                abs(w.lane_id - ego_wp.lane_id)))
        wp_candidates.extend(cands)

    for _ in range(max(1, int(retries))):
        for wp in wp_candidates:
            actor = world.try_spawn_actor(random.choice(bps), wp.transform)
            if actor is not None:
                actor.set_autopilot(False)
                return actor
        world.tick(); time.sleep(0.02)

    raise RuntimeError("Could not spawn or find a target vehicle ahead")

## Spawn veicolo

In [31]:
vehicle = spawn_vehicle(world)
vehicle.set_autopilot(True)  # Disattiva autopilot per controllo manuale

# Fai qualche tick per far stabilizzare il veicolo
for _ in range(10):
    world.tick()
    time.sleep(0.05)

## Spawn target

In [17]:
target = None

# Spawna il target davanti
try:
    target = spawn_vehicle_ahead(world, vehicle, distances=(70.0, 20.0, 40.0), retries=15)
    print(f"Target spawned successfully")
except RuntimeError as e:
    target = None
    print(f"Target spawn failed: {e}")

# Se è spawnato, fermalo in posizione
if target is not None:
    #target.set_autopilot(True)
    target.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0, hand_brake=True))
    print("Target stopped with hand brake")

Target spawned successfully
Target stopped with hand brake


## Collision trigger

In [32]:

# ── Collision trigger ──────────────────────────────────────────────────────────
collision_event       = threading.Event()
collision_info        = {}
flusher_stop          = threading.Event()
stop_loop             = threading.Event()   # fermato da _on_collision
collision_frame_ready = threading.Event()

def _on_collision(event):
    other   = getattr(event, "other_actor", None)
    impulse = getattr(event, "normal_impulse", None)
    collision_info.clear()
    collision_info.update({
        "other_actor": str(other.type_id) if other else "unknown",
        "impulse":     str(impulse),
        "ts":          time.time(),
    })
    collision_event.set()
    stop_loop.set()

# ── Near miss trigger ──────────────────────────────────────────────────────────
NEAR_MISS_THRESHOLD_M = 8.0   # distanza (m) sotto cui scatta il near miss

near_miss_event       = threading.Event()
near_miss_info        = {}
near_miss_frame_ready = threading.Event()

def check_near_miss(vehicle, target, threshold=None):
    """Ritorna la distanza se sotto soglia, altrimenti None."""
    if target is None:
        return None
    dist = vehicle.get_location().distance(target.get_location())
    t = threshold if threshold is not None else NEAR_MISS_THRESHOLD_M
    return dist if dist < t else None

# ── Manual trigger ─────────────────────────────────────────────────────────────
manual_event       = threading.Event()
manual_info        = {}
manual_frame_ready = threading.Event()

def manual_trigger(reason="user_request"):
    """
    Avvia manualmente il flush del ring buffer.
    Può essere chiamato:
      - da un'altra cella: manual_trigger("my reason")
      - via MQTT: pubblica su 'recorder/control' il payload {"cmd": "flush"}
        (il loop di simulazione chiama questa funzione automaticamente)
    """
    manual_info.clear()
    manual_info.update({"reason": reason, "ts": time.time()})
    manual_event.set()
    manual_frame_ready.set()
    print(f"[Manual trigger] flush avviato: {reason}")

# ── Flusher unificato (collisione + near miss + manuale) ──────────────────────
def flusher_thread(out_dir_base="recordings", retention_seconds=30):
    """Reagisce a collision_event, near_miss_event e manual_event."""
    while not flusher_stop.is_set():
        trigger_type   = None
        frame_ready_ev = None
        info           = {}

        # Poll i tre eventi
        while not flusher_stop.is_set():
            if collision_event.wait(timeout=0.05):
                collision_event.clear()
                trigger_type   = "collision"
                frame_ready_ev = collision_frame_ready
                info           = collision_info.copy()
                break
            if near_miss_event.wait(timeout=0.05):
                near_miss_event.clear()
                trigger_type   = "near_miss"
                frame_ready_ev = near_miss_frame_ready
                info           = near_miss_info.copy()
                break
            if manual_event.wait(timeout=0.05):
                manual_event.clear()
                trigger_type   = "manual"
                frame_ready_ev = manual_frame_ready
                info           = manual_info.copy()
                break

        if flusher_stop.is_set() or trigger_type is None:
            break

        frame_ready_ev.wait(timeout=2.0)
        frame_ready_ev.clear()

        now      = time.time()
        cutoff   = now - float(retention_seconds)
        snapshot = [r for r in list(ring_buffer)
                    if getattr(r, "timestamp_wall", now) >= cutoff]
        if not snapshot:
            snapshot = list(ring_buffer)[-1:]

        session_dir = f"{out_dir_base}_{trigger_type}_{time.strftime('%Y%m%dT%H%M%S')}"
        os.makedirs(session_dir, exist_ok=True)

        with open(os.path.join(session_dir, "events.jsonl"), "w", encoding="utf-8") as f:
            for rec in snapshot:
                try:
                    json.dump(rec.to_dict(), f, default=str)
                except Exception:
                    json.dump(str(rec), f, default=str)
                f.write("\n")

        metadata = {
            "trigger":           trigger_type,
            "trigger_info":      info,
            "n_frames":          len(snapshot),
            "saved_at":          now,
            "retention_seconds": retention_seconds,
        }
        with open(os.path.join(session_dir, "metadata.json"), "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2, default=str)

        # Cleanup sessioni più vecchie di retention_seconds
        try:
            for name in os.listdir(os.getcwd()):
                if not name.startswith(out_dir_base + "_"):
                    continue
                dirpath = os.path.join(os.getcwd(), name)
                if os.path.isdir(dirpath) and now - os.path.getmtime(dirpath) > retention_seconds:
                    shutil.rmtree(dirpath, ignore_errors=True)
        except Exception:
            pass

        print(f"Flusher [{trigger_type}]: salvati {len(snapshot)} frame → {session_dir}")


## Avvio sensore collisione e flusher

In [33]:
# --- Avvio sensore collisione e flusher ---
collision_sensor = None
try:
    col_bp = world.get_blueprint_library().find("sensor.other.collision")
    collision_sensor = world.spawn_actor(col_bp, carla.Transform(), attach_to=vehicle)
    collision_sensor.listen(_on_collision)
    flusher = threading.Thread(target=flusher_thread, args=("recordings",), daemon=True)
    flusher.start()
    print("Collision sensor e flusher avviati")
except Exception as e:
    print("Avvio collision sensor/flusher fallito:", e)

Collision sensor e flusher avviati


## Loop Near Miss
Esegui questa cella per la simulazione di near miss.  
Il flusher salva il ring buffer quando la distanza dal target scende sotto `NEAR_MISS_THRESHOLD_M`.  
Imposta `USE_AUTOPILOT = True` per usare il pilota automatico CARLA, o `False` per un controllo gas/freno fisso.


In [19]:
USE_AUTOPILOT = False
THROTTLE      = 0.6   # usato solo se USE_AUTOPILOT = False
BRAKE         = 0.0
STEER         = 0.0
# ──────────────────────────────────────────────────────────────────────────────

frame_id          = 0
mqtt_client       = None
_near_miss_fired  = False

try:
    stop_loop.clear()
    near_miss_event.clear()
    near_miss_frame_ready.clear()
    _near_miss_fired = False

    mqtt_client = start_mqtt(broker="localhost", port=2001)

    if USE_AUTOPILOT:
        vehicle.set_autopilot(True)
        print("Near miss loop avviato — autopilot ATTIVO")
    else:
        vehicle.set_autopilot(False)
        print(f"Near miss loop avviato — throttle={THROTTLE}, brake={BRAKE}")

    while True:
        world.tick()
        msgs = drain_mqtt_queue()

        if not USE_AUTOPILOT:
            vehicle.apply_control(carla.VehicleControl(
                throttle=THROTTLE, brake=BRAKE, steer=STEER))

        record = EventRecord(
            frame_id=frame_id,
            timestamp_sim=world.get_snapshot().timestamp.elapsed_seconds,
            timestamp_wall=time.time(),
            vehicle_state=build_vehicle_state(vehicle),
            camera_frames={},
            warnings=[],
            mqtt_messages=msgs,
        )
        ring_buffer.append(record)
        frame_id += 1

        # ── Near miss check ────────────────────────────────────────────────────
        dist = check_near_miss(vehicle, target)
        if dist is not None and not _near_miss_fired:
            _near_miss_fired = True
            near_miss_info.clear()
            near_miss_info.update({
                "distance_m":  round(dist, 3),
                "threshold_m": NEAR_MISS_THRESHOLD_M,
                "ts":          time.time(),
            })
            near_miss_event.set()
            near_miss_frame_ready.set()
            print(f"[Near miss] distanza {dist:.2f} m al frame {frame_id} — flush avviato")
            time.sleep(1.5)
            break

        # ── Collisione inattesa ────────────────────────────────────────────────
        if stop_loop.is_set():
            collision_frame_ready.set()
            print(f"Collisione inattesa al frame {frame_id} — interruzione loop")
            time.sleep(1.5)
            break

        move_spectator_to(vehicle.get_transform(), spectator, distance=8.0, z=3.0, pitch=-20)
        time.sleep(0.05)

except KeyboardInterrupt:
    print("Interrotto manualmente")
finally:
    vehicle.apply_control(carla.VehicleControl(throttle=0.0, brake=1.0, hand_brake=True))
    #vehicle.set_autopilot(False)
    stop_mqtt(mqtt_client)


c:\anaconda3\envs\carla-env\lib\site-packages\ipykernel_launcher.py:32: DeprecationWarning: Callback API version 1 is deprecated, update to latest version


Near miss loop avviato — throttle=0.6, brake=0.0
[Near miss] distanza 7.72 m al frame 43 — flush avviato


## Loop Collisione
Esegui questa cella per la simulazione di collisione.  
Il flusher salva il ring buffer al momento dell'impatto rilevato dal sensore di collisione.  
Imposta `USE_AUTOPILOT = True` per usare il pilota automatico CARLA, o `False` per un controllo gas/freno fisso (default: dritto contro il target a throttle costante).


In [ ]:
USE_AUTOPILOT = False
THROTTLE      = 0.3   # usato solo se USE_AUTOPILOT = False
BRAKE         = 0.0
STEER         = 0.0
# ──────────────────────────────────────────────────────────────────────────────

frame_id    = 0
mqtt_client = None

try:
    stop_loop.clear()
    collision_event.clear()
    collision_frame_ready.clear()

    mqtt_client = start_mqtt(broker="localhost", port=2001)

    if USE_AUTOPILOT:
        vehicle.set_autopilot(True)
        print("Collision loop avviato — autopilot ATTIVO")
    else:
        vehicle.set_autopilot(False)
        print(f"Collision loop avviato — throttle={THROTTLE}, brake={BRAKE}")

    while True:
        world.tick()
        msgs = drain_mqtt_queue()

        if not USE_AUTOPILOT:
            vehicle.apply_control(carla.VehicleControl(
                throttle=THROTTLE, brake=BRAKE, steer=STEER))

        record = EventRecord(
            frame_id=frame_id,
            timestamp_sim=world.get_snapshot().timestamp.elapsed_seconds,
            timestamp_wall=time.time(),
            vehicle_state=build_vehicle_state(vehicle),
            camera_frames={},
            warnings=[],
            mqtt_messages=msgs,
        )
        ring_buffer.append(record)
        frame_id += 1

        if stop_loop.is_set():
            collision_frame_ready.set()
            print(f"Collisione rilevata al frame {frame_id} — stop simulazione, attendo flush…")
            time.sleep(1.5)
            break

        move_spectator_to(vehicle.get_transform(), spectator, distance=8.0, z=3.0, pitch=-20)
        time.sleep(0.05)

except KeyboardInterrupt:
    print("Interrotto manualmente")
finally:
    vehicle.set_autopilot(False)
    stop_mqtt(mqtt_client)


c:\anaconda3\envs\carla-env\lib\site-packages\ipykernel_launcher.py:32: DeprecationWarning: Callback API version 1 is deprecated, update to latest version


Collision loop avviato — throttle=0.7, brake=0.0
Collisione rilevata al frame 2 — stop simulazione, attendo flush…
Flusher [collision]: salvati 36 frame → recordings_collision_20260515T121720


## Loop Manual Trigger
Esegui questa cella per avviare un loop in cui il flush può essere attivato in due modi:
- **Programmatico**: da un'altra cella chiama `manual_trigger("descrizione")` mentre questo loop gira.
- **MQTT**: pubblica `{"cmd": "flush", "reason": "descrizione"}` sul topic `recorder/control`.

Il loop si interrompe con `KeyboardInterrupt` (pulsante ■ del notebook).


In [34]:
USE_AUTOPILOT = True
THROTTLE      = 0.3   # usato solo se USE_AUTOPILOT = False
BRAKE         = 0.0
STEER         = 0.0
# ──────────────────────────────────────────────────────────────────────────────

frame_id    = 0
mqtt_client = None
_kb_hooked  = False
manual_stop = threading.Event()

try:
    manual_event.clear()
    manual_frame_ready.clear()
    manual_stop.clear()

    mqtt_client = start_mqtt(broker="localhost", port=2001)

    # ── Listener tastiera per tasto M e S ─────────────────────────────────────
    try:
        import keyboard as _keyboard
        _keyboard.on_press_key("m", lambda e: manual_trigger("keyboard_M"))
        _keyboard.on_press_key("s", lambda e: manual_stop.set())
        _kb_hooked = True
    except Exception as _kb_err:
        print(f"  (keyboard non disponibile, tasti M e S disabilitati: {_kb_err})")

    if USE_AUTOPILOT:
        vehicle.set_autopilot(True)
        print("Manual trigger loop avviato — autopilot ATTIVO")
    else:
        vehicle.set_autopilot(False)
        print(f"Manual trigger loop avviato — throttle={THROTTLE}, brake={BRAKE}")
    print("  • premi M sulla tastiera per attivare il flush")
    print("  • premi S sulla tastiera per stoppare la simulazione")
    print("  • pubblica su recorder/control: {\"cmd\": \"flush\", \"reason\": \"...\"}")
    print("  • oppure chiama manual_trigger() da un'altra cella")

    while True:
        world.tick()
        msgs = drain_mqtt_queue()

        if not USE_AUTOPILOT:
            vehicle.apply_control(carla.VehicleControl(
                throttle=THROTTLE, brake=BRAKE, steer=STEER))

        record = EventRecord(
            frame_id=frame_id,
            timestamp_sim=world.get_snapshot().timestamp.elapsed_seconds,
            timestamp_wall=time.time(),
            vehicle_state=build_vehicle_state(vehicle),
            camera_frames={},
            warnings=[],
            mqtt_messages=msgs,
        )
        ring_buffer.append(record)
        frame_id += 1

        # ── Manual trigger via MQTT ────────────────────────────────────────────
        for msg in msgs:
            if msg.get("topic") == "recorder/control":
                payload = msg.get("payload", {})
                if isinstance(payload, dict) and payload.get("cmd") == "flush":
                    reason = payload.get("reason", "mqtt_flush")
                    manual_trigger(reason)

        # ── Stop manuale via tasto S ───────────────────────────────────────────
        if manual_stop.is_set():
            print(f"Stop manuale al frame {frame_id} — interruzione loop")
            break

        # ── Collisione inattesa ────────────────────────────────────────────────
        if stop_loop.is_set():
            collision_frame_ready.set()
            print(f"Collisione inattesa al frame {frame_id} — interruzione loop")
            time.sleep(1.5)
            break

        move_spectator_to(vehicle.get_transform(), spectator, distance=8.0, z=3.0, pitch=-20)
        time.sleep(0.05)

except KeyboardInterrupt:
    print("Interrotto manualmente")
finally:
    if _kb_hooked:
        try:
            _keyboard.unhook_all()
        except Exception:
            pass
    vehicle.set_autopilot(False)
    stop_mqtt(mqtt_client)


c:\anaconda3\envs\carla-env\lib\site-packages\ipykernel_launcher.py:32: DeprecationWarning: Callback API version 1 is deprecated, update to latest version


Manual trigger loop avviato — autopilot ATTIVO
  • premi M sulla tastiera per attivare il flush
  • premi S sulla tastiera per stoppare la simulazione
  • pubblica su recorder/control: {"cmd": "flush", "reason": "..."}
  • oppure chiama manual_trigger() da un'altra cella
[Manual trigger] flush avviato: keyboard_M
Flusher [manual]: salvati 104 frame → recordings_manual_20260519T121110
Stop manuale al frame 148 — interruzione loop


In [35]:
settings.synchronous_mode = False
world.apply_settings(settings)
print("Modalità sincrona disattivata")

Modalità sincrona disattivata


In [36]:
# --- Teardown: flusher + collision sensor + veicolo ---
try:
    flusher_stop.set()
except NameError:
    pass

if "collision_sensor" in dir() and collision_sensor is not None:
    try:
        collision_sensor.stop()
    except Exception:
        pass
    try:
        collision_sensor.destroy()
    except Exception:
        pass
    collision_sensor = None

# Distruggi il target se spawned
if target is not None:
    try:
        target.destroy()
    except Exception:
        pass
    target = None

vehicle.destroy()
print("Veicolo distrutto")
vehicle = None

Veicolo distrutto
